<a href="https://colab.research.google.com/github/chawlaniharika23/Fraud_Detection_Research/blob/main/notebooks/03_improved_feature_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03 — Leakage-Safe Feature Engineering and Feature Selection

## Financial Fraud Detection: IEEE-CIS

### Objective
Transform the raw IEEE-CIS train, validation, and test partitions into model-ready sparse matrices, then select the 100 most informative features using a reproducible tree-based ranking procedure.

### Improvements over the original notebook
1. **Training-only preprocessing:** frequency maps, imputers, one-hot encoder, and scaler are fitted on the training partition only.
2. **No test leakage:** validation and test data are transformed using the saved training-fitted objects; neither partition is used to rank features.
3. **More robust feature ranking:** Random Forest and Extra Trees provide complementary importance rankings. Their percentile ranks are aggregated rather than relying on one estimator's raw impurity scores.
4. **Class-imbalance awareness:** both ranking estimators use `class_weight="balanced_subsample"`.
5. **Memory-aware processing:** each split is loaded and transformed separately; temporary DataFrames are deleted after transformation.
6. **Downstream compatibility:** the notebook saves the exact `ieee_X_{split}_selected_100.npz` files expected by Notebook 04.

### Methodological note
Tree impurity importance can be biased toward high-cardinality or correlated variables. Rank aggregation across two tree ensembles is a practical robustness improvement, not a guarantee of unbiased feature importance. The final feature count remains 100 for compatibility with the current downstream notebook. Alternative feature counts can be studied as a separate ablation experiment.

## 1. Imports and reproducibility

In [ ]:
import os
import gc
import json
import time
import joblib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy.sparse import csr_matrix, hstack, save_npz, load_npz

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

RANDOM_STATE = 42
TARGET_FEATURE_COUNT = 100
RANKING_SAMPLE_SIZE = 50000

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Feature selection notebook initialized.")

## 2. Mount Google Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/fraud-detection-research")
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RECOVERY_DIR = PROCESSED_DIR / "recovered"
REPORT_DIR = PROCESSED_DIR / "reports"

RECOVERY_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

IEEE_TRAIN_X = PROCESSED_DIR / "ieee_X_train.parquet"
IEEE_VAL_X = PROCESSED_DIR / "ieee_X_validation.parquet"
IEEE_TEST_X = PROCESSED_DIR / "ieee_X_test.parquet"

IEEE_TRAIN_Y = PROCESSED_DIR / "ieee_y_train.parquet"
IEEE_VAL_Y = PROCESSED_DIR / "ieee_y_validation.parquet"
IEEE_TEST_Y = PROCESSED_DIR / "ieee_y_test.parquet"

required_files = [
    IEEE_TRAIN_X, IEEE_VAL_X, IEEE_TEST_X,
    IEEE_TRAIN_Y, IEEE_VAL_Y, IEEE_TEST_Y
]
missing_files = [str(p) for p in required_files if not p.exists()]
if missing_files:
    raise FileNotFoundError(
        "Required Notebook 02 outputs are missing:\n" + "\n".join(missing_files)
        + "\nRun Notebook 02 successfully before running Notebook 03."
    )

print("Processed data directory:", PROCESSED_DIR)
print("Feature-selection artifact directory:", RECOVERY_DIR)

## 3. Helper functions

The same transformation function is applied to each split. It uses only mappings and fitted transformers learned from the training partition.

In [ ]:
def load_target(path):
    y_frame = pd.read_parquet(path)
    if "isFraud" not in y_frame.columns:
        raise ValueError(f"Target file {path.name} does not contain 'isFraud'.")
    y = y_frame["isFraud"].to_numpy(dtype=np.int8)
    if not np.isin(y, [0, 1]).all():
        raise ValueError(f"Target file {path.name} contains values other than 0/1.")
    return y


def make_one_hot_encoder():
    # Support both newer and older scikit-learn versions.
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=True,
            dtype=np.float32
        )


def apply_frequency_encoding(frame, columns, mappings):
    frame = frame.copy()
    for col in columns:
        frame[col] = (
            frame[col].map(mappings[col]).fillna(0).astype(np.float32)
        )
    return frame


def transform_split(frame, numeric_features, low_cardinality_cols,
                    frequency_maps, numeric_imputer, scaler,
                    categorical_imputer, encoder):
    # Frequency-encode high-cardinality columns using training-only mappings.
    frame = apply_frequency_encoding(
        frame, list(frequency_maps.keys()), frequency_maps
    )

    numeric_array = numeric_imputer.transform(
        frame[numeric_features]
    ).astype(np.float32, copy=False)
    numeric_array = scaler.transform(numeric_array).astype(
        np.float32, copy=False
    )

    if low_cardinality_cols:
        categorical_array = categorical_imputer.transform(
            frame[low_cardinality_cols]
        )
        categorical_sparse = encoder.transform(categorical_array).tocsr()
    else:
        categorical_sparse = csr_matrix((len(frame), 0), dtype=np.float32)

    numeric_sparse = csr_matrix(numeric_array, dtype=np.float32)
    transformed = hstack(
        [numeric_sparse, categorical_sparse],
        format="csr",
        dtype=np.float32
    )

    del numeric_array, numeric_sparse, categorical_sparse, frame
    gc.collect()
    return transformed

## 4. Load training partition and inspect feature types

In [ ]:
X_train_raw = pd.read_parquet(IEEE_TRAIN_X)
y_train = load_target(IEEE_TRAIN_Y)

if len(X_train_raw) != len(y_train):
    raise ValueError("Training feature and target row counts do not match.")

numeric_cols = X_train_raw.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X_train_raw.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

low_cardinality_cols = [
    col for col in categorical_cols
    if X_train_raw[col].nunique(dropna=True) <= 50
]
high_cardinality_cols = [
    col for col in categorical_cols
    if X_train_raw[col].nunique(dropna=True) > 50
]

numeric_features = numeric_cols + high_cardinality_cols

print("Training shape:", X_train_raw.shape)
print("Fraud cases:", int(y_train.sum()))
print("Fraud prevalence:", f"{y_train.mean() * 100:.4f}%")
print("Numeric columns:", len(numeric_cols))
print("Low-cardinality categorical columns:", len(low_cardinality_cols))
print("High-cardinality categorical columns:", len(high_cardinality_cols))

if not numeric_features and not low_cardinality_cols:
    raise ValueError("No usable feature columns were found.")

## 5. Fit training-only frequency encoding and imputers

High-cardinality categorical values are represented by their relative frequency in the training partition. Unseen validation/test categories map to zero. Numerical missing values are imputed using training medians; low-cardinality categorical missing values are imputed using the training most-frequent category.

In [ ]:
frequency_maps = {}
for col in high_cardinality_cols:
    frequency_maps[col] = X_train_raw[col].value_counts(
        normalize=True,
        dropna=True
    ).to_dict()

X_train_work = apply_frequency_encoding(
    X_train_raw, high_cardinality_cols, frequency_maps
)

numeric_imputer = SimpleImputer(
    strategy="median",
    keep_empty_features=True
)
numeric_imputer.fit(X_train_work[numeric_features])

if low_cardinality_cols:
    categorical_imputer = SimpleImputer(strategy="most_frequent")
    categorical_imputer.fit(X_train_work[low_cardinality_cols])
    encoder = make_one_hot_encoder()
    X_train_cat_imputed = categorical_imputer.transform(
        X_train_work[low_cardinality_cols]
    )
    encoder.fit(X_train_cat_imputed)
else:
    categorical_imputer = None
    encoder = None
    X_train_cat_imputed = None

print("Training-only frequency maps:", len(frequency_maps))
print("Numerical imputer fitted:", numeric_imputer is not None)
print("Categorical columns fitted:", len(low_cardinality_cols))

## 6. Transform training data and save preprocessing objects

In [ ]:
start_time = time.time()

X_train_numeric = numeric_imputer.transform(
    X_train_work[numeric_features]
).astype(np.float32, copy=False)

scaler = StandardScaler()
X_train_numeric = scaler.fit_transform(
    X_train_numeric
).astype(np.float32, copy=False)

if low_cardinality_cols:
    X_train_cat_imputed = categorical_imputer.transform(
        X_train_work[low_cardinality_cols]
    )
    X_train_cat_encoded = encoder.transform(X_train_cat_imputed).tocsr()
else:
    X_train_cat_encoded = csr_matrix(
        (len(X_train_work), 0), dtype=np.float32
    )

X_train_processed = hstack(
    [csr_matrix(X_train_numeric), X_train_cat_encoded],
    format="csr",
    dtype=np.float32
)

feature_names = (
    numeric_features
    + (
        encoder.get_feature_names_out(low_cardinality_cols).tolist()
        if low_cardinality_cols else []
    )
)

if X_train_processed.shape[1] != len(feature_names):
    raise ValueError("Processed feature count does not match feature-name count.")

save_npz(
    RECOVERY_DIR / "ieee_X_train_processed.npz",
    X_train_processed,
    compressed=True
)

joblib.dump(frequency_maps, RECOVERY_DIR / "frequency_maps.joblib")
joblib.dump(numeric_imputer, RECOVERY_DIR / "numeric_imputer.joblib")
joblib.dump(scaler, RECOVERY_DIR / "numeric_scaler.joblib")
if categorical_imputer is not None:
    joblib.dump(categorical_imputer, RECOVERY_DIR / "categorical_imputer.joblib")
if encoder is not None:
    joblib.dump(encoder, RECOVERY_DIR / "categorical_encoder.joblib")

pd.Series(feature_names, name="feature_name").to_csv(
    RECOVERY_DIR / "ieee_feature_names.csv", index=False
)

print("Processed training matrix:", X_train_processed.shape)
print("Non-zero entries:", X_train_processed.nnz)
print("Density:", f"{X_train_processed.nnz / (X_train_processed.shape[0] * X_train_processed.shape[1]):.6f}")
print("Training transformation time:", f"{time.time() - start_time:.2f} seconds")

## 7. Release training intermediates

The saved sparse matrix and fitted transformers are retained. Raw and dense intermediate arrays are removed before validation/test transformation to reduce peak memory use.

In [ ]:
del X_train_raw, X_train_work, X_train_numeric
if X_train_cat_imputed is not None:
    del X_train_cat_imputed
del X_train_cat_encoded
gc.collect()

print("Training intermediate DataFrames and dense arrays released.")

## 8. Transform validation and test partitions

In [ ]:
for split_name, x_path in [
    ("validation", IEEE_VAL_X),
    ("test", IEEE_TEST_X)
]:
    print(f"\nProcessing {split_name} partition...")
    X_split_raw = pd.read_parquet(x_path)
    y_split = load_target(
        IEEE_VAL_Y if split_name == "validation" else IEEE_TEST_Y
    )

    if len(X_split_raw) != len(y_split):
        raise ValueError(f"{split_name} feature and target row counts do not match.")

    X_split_processed = transform_split(
        X_split_raw,
        numeric_features,
        low_cardinality_cols,
        frequency_maps,
        numeric_imputer,
        scaler,
        categorical_imputer,
        encoder
    )

    if X_split_processed.shape[1] != len(feature_names):
        raise ValueError(f"{split_name} feature count differs from training.")

    save_npz(
        RECOVERY_DIR / f"ieee_X_{split_name}_processed.npz",
        X_split_processed,
        compressed=True
    )

    print(
        f"{split_name}: {X_split_processed.shape}, "
        f"fraud rate={y_split.mean() * 100:.4f}%"
    )

    del X_split_processed, y_split
    gc.collect()

print("Validation and test transformations completed.")

## 9. Verify processed matrices and feature alignment

This verification reads sparse matrices one at a time. The feature names are shared across all partitions because the encoder was fitted only on training data.

In [ ]:
from scipy.sparse import load_npz

matrix_shapes = {}
for split_name in ["train", "validation", "test"]:
    matrix_path = RECOVERY_DIR / f"ieee_X_{split_name}_processed.npz"
    if not matrix_path.exists():
        raise FileNotFoundError(f"Missing processed matrix: {matrix_path}")

    matrix = load_npz(matrix_path)
    matrix_shapes[split_name] = matrix.shape

    assert matrix.shape[1] == len(feature_names)
    assert matrix.dtype == np.float32

    print(
        f"{split_name}: shape={matrix.shape}, "
        f"nnz={matrix.nnz:,}, dtype={matrix.dtype}"
    )
    del matrix
    gc.collect()

assert matrix_shapes["train"][1] == matrix_shapes["validation"][1] == matrix_shapes["test"][1]
print("Processed feature dimensions are aligned.")

## 10. Prepare a stratified training sample for feature ranking

The ranking estimators are trained on a stratified sample of the training partition to limit computational cost while preserving the minority-class proportion. Validation and test labels are not used in this stage.

In [ ]:
y_train = load_target(IEEE_TRAIN_Y)

sample_size = min(RANKING_SAMPLE_SIZE, len(y_train))
sample_indices, _ = train_test_split(
    np.arange(len(y_train)),
    train_size=sample_size,
    stratify=y_train,
    random_state=RANDOM_STATE
)

X_train_full = load_npz(
    RECOVERY_DIR / "ieee_X_train_processed.npz"
)
X_rank_sample = X_train_full[sample_indices]
y_rank_sample = y_train[sample_indices]

print("Full training matrix:", X_train_full.shape)
print("Feature-ranking sample:", X_rank_sample.shape)
print("Fraud cases in sample:", int(y_rank_sample.sum()))
print("Fraud percentage in sample:", f"{y_rank_sample.mean() * 100:.4f}%")

## 11. Fit complementary tree-based feature rankers

Random Forest and Extra Trees capture nonlinear feature interactions using different split-selection mechanisms. Both use balanced subsampling to reduce the influence of the majority class during tree construction.

The feature rankings are combined using normalized percentile ranks. This reduces dependence on the absolute scale of one estimator's impurity-importance values.

In [ ]:
ranker_configs = {
    "RandomForest": RandomForestClassifier(
        n_estimators=120,
        max_depth=14,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=120,
        max_depth=14,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE + 1,
        n_jobs=-1
    )
}

importance_scores = {}
ranker_times = {}

for model_name, ranker in ranker_configs.items():
    print(f"Training {model_name} feature ranker...")
    start = time.time()
    ranker.fit(X_rank_sample, y_rank_sample)
    ranker_times[model_name] = time.time() - start
    importance_scores[model_name] = ranker.feature_importances_.astype(np.float64)
    print(
        f"{model_name} completed in {ranker_times[model_name]:.2f}s; "
        f"features ranked: {len(importance_scores[model_name])}"
    )
    joblib.dump(ranker, RECOVERY_DIR / f"{model_name.lower()}_feature_ranker.joblib")
    gc.collect()

print("Both feature-ranking models trained.")

## 12. Aggregate importance rankings and select top 100 features

In [ ]:
importance_df = pd.DataFrame({
    "Feature_Index": np.arange(len(feature_names), dtype=np.int64),
    "Feature": feature_names,
    "RandomForest_Importance": importance_scores["RandomForest"],
    "ExtraTrees_Importance": importance_scores["ExtraTrees"]
})

# Convert each model's importance into a percentile rank in [0, 1].
for col in ["RandomForest_Importance", "ExtraTrees_Importance"]:
    importance_df[col + "_Rank"] = (
        importance_df[col].rank(method="average", ascending=True)
        / len(importance_df)
    )

importance_df["Aggregated_Rank_Score"] = importance_df[
    ["RandomForest_Importance_Rank", "ExtraTrees_Importance_Rank"]
].mean(axis=1)

importance_df = importance_df.sort_values(
    "Aggregated_Rank_Score", ascending=False
).reset_index(drop=True)

if len(importance_df) < TARGET_FEATURE_COUNT:
    raise ValueError(
        f"Only {len(importance_df)} features available; "
        f"cannot select {TARGET_FEATURE_COUNT}."
    )

top_100_indices = importance_df.head(TARGET_FEATURE_COUNT).index.to_numpy()
selected_feature_names = importance_df.head(
    TARGET_FEATURE_COUNT
)["Feature"].tolist()

# These are column positions in the original processed matrix, not row indices.
selected_column_indices = importance_df.head(
    TARGET_FEATURE_COUNT
)["Feature_Index"].to_numpy(dtype=np.int64)

importance_df.to_csv(
    RECOVERY_DIR / "ieee_feature_importance_ranking.csv",
    index=False
)
pd.Series(selected_feature_names, name="feature_name").to_csv(
    RECOVERY_DIR / "ieee_top_100_features.csv",
    index=False
)
np.save(
    RECOVERY_DIR / "ieee_top_100_feature_indices.npy",
    selected_column_indices
)

print("Selected features:", len(selected_feature_names))
display(importance_df.head(20)[[
    "Feature",
    "RandomForest_Importance",
    "ExtraTrees_Importance",
    "Aggregated_Rank_Score"
]])

## 13. Feature-ranker agreement and importance visualization

The overlap between the individual top-100 lists provides a simple diagnostic of ranking agreement. Low overlap indicates that feature importance is sensitive to the estimator and should be discussed as a limitation.

In [ ]:
rf_top = set(
    np.argsort(importance_scores["RandomForest"])[-TARGET_FEATURE_COUNT:]
)
et_top = set(
    np.argsort(importance_scores["ExtraTrees"])[-TARGET_FEATURE_COUNT:]
)
top_overlap = len(rf_top.intersection(et_top))

print(f"Random Forest / Extra Trees top-{TARGET_FEATURE_COUNT} overlap: {top_overlap}")
print(f"Jaccard similarity: {top_overlap / max(len(rf_top.union(et_top)), 1):.4f}")
print("Ranker training times (seconds):", ranker_times)

import matplotlib.pyplot as plt

top20_plot = importance_df.head(20).sort_values(
    "Aggregated_Rank_Score", ascending=True
)

plt.figure(figsize=(11, 8))
plt.barh(
    top20_plot["Feature"],
    top20_plot["Aggregated_Rank_Score"]
)
plt.xlabel("Aggregated percentile-rank score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 14. Apply the same selected features to all partitions

In [ ]:
# Persist the same selected column positions for all splits.
for split_name in ["train", "validation", "test"]:
    source_path = RECOVERY_DIR / f"ieee_X_{split_name}_processed.npz"
    output_path = RECOVERY_DIR / f"ieee_X_{split_name}_selected_100.npz"

    full_matrix = load_npz(source_path)
    if full_matrix.shape[1] != len(feature_names):
        raise ValueError(
            f"{split_name} has {full_matrix.shape[1]} columns; "
            f"expected {len(feature_names)}."
        )

    selected_matrix = full_matrix[:, selected_column_indices].tocsr()
    save_npz(output_path, selected_matrix, compressed=True)

    print(
        f"{split_name}: {full_matrix.shape} -> {selected_matrix.shape}; "
        f"saved {output_path.name}"
    )

    del full_matrix, selected_matrix
    gc.collect()

## 15. Final output validation and handoff to Notebook 04

In [ ]:
from scipy.sparse import load_npz

final_rows = []
for split_name, y_path in [
    ("train", IEEE_TRAIN_Y),
    ("validation", IEEE_VAL_Y),
    ("test", IEEE_TEST_Y)
]:
    matrix_path = RECOVERY_DIR / f"ieee_X_{split_name}_selected_100.npz"
    matrix = load_npz(matrix_path)
    y_split = load_target(y_path)

    assert matrix.shape[0] == len(y_split)
    assert matrix.shape[1] == TARGET_FEATURE_COUNT

    final_rows.append({
        "Split": split_name,
        "Rows": matrix.shape[0],
        "Selected_Features": matrix.shape[1],
        "Fraud_Count": int(y_split.sum()),
        "Fraud_Rate_Percent": float(y_split.mean() * 100),
        "Sparse_NNZ": int(matrix.nnz)
    })

    print(f"{split_name}: {matrix.shape}, labels={len(y_split)}")
    del matrix, y_split
    gc.collect()

final_summary = pd.DataFrame(final_rows)
display(final_summary.round(4))
final_summary.to_csv(
    REPORT_DIR / "feature_selection_output_summary.csv",
    index=False
)

# Save a compact manifest for reproducibility.
manifest = {
    "dataset": "IEEE-CIS",
    "preprocessing_fit_partition": "train",
    "feature_selection_fit_partition": "stratified sample from train",
    "feature_selection_method": "mean percentile-rank aggregation of Random Forest and Extra Trees impurity importances",
    "ranking_sample_size_requested": RANKING_SAMPLE_SIZE,
    "selected_feature_count": TARGET_FEATURE_COUNT,
    "random_state": RANDOM_STATE,
    "feature_names_file": "recovered/ieee_feature_names.csv",
    "selected_features_file": "recovered/ieee_top_100_features.csv",
    "selected_indices_file": "recovered/ieee_top_100_feature_indices.npy",
    "selected_matrices": {
        split: f"recovered/ieee_X_{split}_selected_100.npz"
        for split in ["train", "validation", "test"]
    },
    "ranker_training_times_seconds": ranker_times,
    "top_100_overlap_between_rankers": int(top_overlap)
}
with open(REPORT_DIR / "ieee_feature_selection_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Notebook 03 completed successfully.")
print("Notebook 04 can now load ieee_X_train_selected_100.npz and ieee_X_validation_selected_100.npz.")